# データの前処理

In [ ]:
# データの読み込み
import pandas as pd
file = 'Boston.csv'
df = pd.read_csv(file)
df.head(3)

In [ ]:

print(f'Head contents of {file} is \n{df.head(5)}')
columns = df.columns
print(f'Label of {file} : {columns}')
label_means = {
    'CRIME':'犯罪発生率',
    'ZN':'25,000平方フィート以上の住居区画の占める割合',
    'INDUS':'小売業以外の商業が占める面積の割合',
    'CHAS':'チャールズ川の近くか否か',
    'NOX':'窒素感化物の濃度',
    'RM':'住居の平均部屋数',
    'AGE':'1940年より前に建てられた物件の割合',
    'DIS':'ボストン市内の5つの雇用施設からの距離',
    'RAD':'環状高速道路へのアクセスしやすさ',
    'TAX':'10,000ドルあたりの不動産税率の総計',
    'PTRATIO':'町ごとの教員一人当たりの児童生徒数',
    'LSTAT':'人口における低所得者の割合',
    'PRICE':'その地域の住宅平均価格'
    }
print('ラベルの意味：')
for key, value in label_means.items():
    print(key,':',value)
print(f'Matrix size : {df.shape}')

In [ ]:
# 欠損値の確認(平均値で欠損値を穴埋めする)
print('欠損値の数を確認\n',df.isnull().sum())

df2 = df.fillna(df.mean(numeric_only=True))
print('穴埋め後の欠損値の数を確認\n',df2.isnull().sum())

In [ ]:
# ダミー変数化
dummy = pd.get_dummies(df2['CRIME'], drop_first=True, dtype=int)
df3 = df2.join(dummy)
df3 = df3.drop(['CRIME'], axis=1)
df3.head(3)

In [ ]:
# データの標準化
from sklearn.preprocessing import StandardScaler
# 中身が整数だとfit_transformで警告になるのでfloat型に変換(省略可能)
df4 = df3.astype('float')
# 標準化
sc = StandardScaler()
sc_df = sc.fit_transform(df4)

# 14.3 主成分分析の実施

## 14.3.1 モジュールのインポート

In [ ]:
from sklearn.decomposition import PCA

## 14.3.2 モデルの作成

In [ ]:
model = PCA(n_components = 2, whiten = True)

### モデルに学習させる

In [ ]:
model.fit(sc_df)

## 第1軸と第2軸の固有ベクトル

In [ ]:
# 新規の第1軸(第一主成分)の固有ベクトル
print(f'vector dimension = {len(model.components_[0])}')
print(model.components_[0])
print('----------')
# 新規の第2軸(第二主成分)の固有ベクトル
print(model.components_[1])


In [ ]:
# 既存のsc_dfを新しい２つの軸に当てはめる(主成分得点の計算)
new = model.transform(sc_df)

new_df = pd.DataFrame(new)
new_df.head(3)

# 14.4 結果の評価

## 14.4.1 主成分負荷量の確認

In [ ]:
# 新しい2列ともとの列を結合

new_df.columns = ['PC1', 'PC2']
# 標準化済みの既存データ(numpy)をデータフレーム化
df5 = pd.DataFrame(sc_df, columns = df4.columns)
# ２つのデータフレームを列方向に結合
df6 = pd.concat([df5, new_df], axis=1)
df6.head(3)

In [ ]:
# 主成分負荷量の計算
df_corr = df6.corr() # 相関係数の計算
df_corr.loc[:'very_low', 'PC1':]

In [ ]:
# 相関係数を大きい順に並べる
pc_corr = df_corr.loc[:'very_low', 'PC1':]
pc_corr['PC1'].sort_values(ascending = False)

In [ ]:
pc_corr['PC2'].sort_values(ascending = False)

In [ ]:
print('ラベルの意味：')
for key, value in label_means.items():
    print(key,':',value)

In [ ]:
# 新しい列の散布図
# 都市の非発展度合いと住環境の良さ
col = ['Countryside', 'Exclusive residential']

new_df.columns = col

new_df.plot(kind = 'scatter', x = col[0], y=col[1])

## 14.4.2 最適な列の個数　ー　寄与率　ー

### 新規の軸をすべて用意する

In [ ]:
model = PCA(whiten=True) # n_components を指定しないと、新規の軸はすべてデフォルトで作られる

# 学習と新規軸へのデータの当てはめを一括で行う
tmp = model.fit_transform(sc_df)
tmp.shape

### 寄与率を表示する

In [ ]:
# model.explained_variance_
model.explained_variance_ratio_

### 累積寄与率

In [ ]:
ratio = model.explained_variance_ratio_ # 寄与率のデータ集合

array = []

for i in range(len(ratio)):
    ruiseki = sum(ratio[0:(i+1)])

    array.append(ruiseki)

pd.Series(array).plot(kind = 'line')

### 情報量の閾値を設定して必要な列の数を求める

In [ ]:
thred = 0.8 # 累積寄与率の閾値
for i in range(len(array)):
    if array[i] > thred:
        print(f'累積寄与率が{thred}を超える新規軸の数は{i+1}')
        break


### 新規の列を5つに設定してモデルに学習させる

In [ ]:
# もとのデータの全情報の80%を賄うために、新規の列を5つに設定
model = PCA(n_components=5, whiten=True)
model.fit(sc_df) # 学習
# 元のデータを新規の列(5列)に当てはめる
new = model.transform(sc_df)

### 5列のデータをcsvファイルに保存

In [ ]:
# 主成分分析の結果をデータフレームに変換
col = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
new_df2 = pd.DataFrame(new, columns = col)

new_df2.to_csv('boston_pca.csv', index = False)